<a href="https://colab.research.google.com/github/Jyotiii49/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()

    return getpass.getpass("Enter Hugging Face READ token: ")

token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [token])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Feature window: February 2026")
print("Label window: March 2026")

Enter Hugging Face READ token: ··········
Feature window: February 2026
Label window: March 2026


**Unit of analysis:** One row represents one content item for one client.

The feature window is February 2026 (2026-02-01 to 2026-02-28). The label window is March 2026 (2026-03-01 to 2026-03-31).

The two windows are deliberately separated so that March outcome information is not used to create February features. This reduces the risk of temporal leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "char_count_tier",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

label = ["ctr"]

context = [
    "content_id",
    "client_id",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

excluded = {
    "ctr": "Target variable; using it as an input feature would leak the outcome.",
    "client_name": "Excluded to avoid client-identifying information.",
    "url": "Excluded because URLs are not required for this ranking-signal analysis."
}

print("Features:", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)

Features: ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'char_count_tier', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']
Label: ['ctr']
Context: ['content_id', 'client_id', 'impression_tier', 'position_tier', 'trend_direction']
Excluded: {'ctr': 'Target variable; using it as an input feature would leak the outcome.', 'client_name': 'Excluded to avoid client-identifying information.', 'url': 'Excluded because URLs are not required for this ranking-signal analysis.'}


### Features

Candidate features describe search demand, competition, content characteristics, engagement and historical trends.

### Label

**CTR (Click-Through Rate)** is the observed outcome to be predicted/ranked.

### Context

`content_id` and `client_id` identify the content and client. Tier and trend fields provide additional context for interpreting the ranking signals.

### Excluded

CTR is excluded from the input features because it is the label. Client names and URLs are excluded because they are unnecessary for the analysis and may expose identifying information.

All predictive features must be information that was knowable before the March outcome window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# ML-04: Verify the February and March windows

feb_check = con.sql(f"""
SELECT
    MIN(report_date) AS feb_start,
    MAX(report_date) AS feb_end,
    COUNT(DISTINCT report_date) AS feb_days,
    COUNT(*) AS feb_rows
FROM {FEB}
""").df()

mar_check = con.sql(f"""
SELECT
    MIN(report_date) AS mar_start,
    MAX(report_date) AS mar_end,
    COUNT(DISTINCT report_date) AS mar_days,
    COUNT(*) AS mar_rows
FROM {MAR}
""").df()

print("FEBRUARY 2026 — FEATURE WINDOW")
display(feb_check)

print("MARCH 2026 — LABEL WINDOW")
display(mar_check)

FEBRUARY 2026 — FEATURE WINDOW


,feb_start,feb_end,feb_days,feb_rows
0,2026-02-01,2026-02-28,28,7355108


MARCH 2026 — LABEL WINDOW


,mar_start,mar_end,mar_days,mar_rows
0,2026-03-01,2026-03-31,31,9841378


In [14]:
# 1. Check the February grain

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS content_client_pairs
FROM {FEB}
""").df()

display(grain_check)

,daily_rows,content_client_pairs
0,7355108,321546


In [15]:
# 2. Check missing values

missing_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_clicks,
    COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS missing_gsc_available
FROM {FEB}
""").df()

display(missing_check)

,total_rows,missing_impressions,missing_clicks,missing_gsc_available
0,7355108,92256,92256,92256


In [16]:
# 3. Check March GSC availability

march_gsc_check = con.sql(f"""
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available) AS measured_rows,
    COUNT(*) FILTER (WHERE NOT gsc_data_available) AS unavailable_rows
FROM {MAR}
""").df()

display(march_gsc_check)

,total_march_rows,measured_rows,unavailable_rows
0,9841378,3611061,6230317


In [17]:
# 4. Check March clicks and impressions

march_totals = con.sql(f"""
SELECT
    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
FROM {MAR}
WHERE gsc_data_available
""").df()

display(march_totals)

,march_impressions,march_clicks,march_ctr
0,280657589.0,821832.0,0.002928


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
data_limits = [
    "Historical data cannot guarantee future ranking performance.",
    "CTR is affected by search position, impressions and other factors, so CTR is not a pure measure of content quality.",
    "The data is observational and supports directional decision-making, not causal conclusions.",
    "GSC data may be unavailable for some periods, so unavailable data must not automatically be treated as zero traffic.",
    "February and March are separate windows; using March information to construct February features would create temporal leakage.",
    "Different clients may have different amounts of historical coverage, so comparisons may not be perfectly balanced."
]

for i, limit in enumerate(data_limits, 1):
    print(f"{i}. {limit}")

1. Historical data cannot guarantee future ranking performance.
2. CTR is affected by search position, impressions and other factors, so CTR is not a pure measure of content quality.
3. The data is observational and supports directional decision-making, not causal conclusions.
4. GSC data may be unavailable for some periods, so unavailable data must not automatically be treated as zero traffic.
5. February and March are separate windows; using March information to construct February features would create temporal leakage.
6. Different clients may have different amounts of historical coverage, so comparisons may not be perfectly balanced.


## Data limits

This dataset supports directional ranking-signal analysis, but it cannot establish causation.

Historical search performance may not represent future performance. CTR is influenced by search position, impressions and other factors, so it should not be interpreted as a pure measure of content quality.

GSC data availability may differ across time and clients. A missing or unavailable observation should not automatically be interpreted as zero traffic.

The February feature window and March outcome window must remain separate. Using information from the March outcome period to construct February features would create temporal leakage.

Therefore, the results should be treated as decision-support and directional evidence rather than a guarantee of future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.